# Predictive Maintenance: Degradation Forecasting & Anomaly Detection

**Read this first**: every dataset in this notebook is **synthetic**. The hydraulic baseline (head loss at a given roughness) is computed via this project's real, verified Darcy-Weisbach engine — but the degradation trend over time, sensor noise, and injected anomalies are all fabricated for demonstration. This notebook shows the ML *pattern* — how you'd structure training and inference for pipe predictive maintenance — not a validated predictive tool. Swap in your own real inspection records / sensor logs for actual use.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np

## Part 1: Roughness Degradation Forecasting

### Generate the synthetic (physics-grounded) dataset

In [2]:
from src.machine_learning.synthetic_data import generate_synthetic_roughness_degradation

df = generate_synthetic_roughness_degradation(
    diameter_m=0.0508, flow_rate_m3s=0.0005, length_m=100.0, days=730,
)
df.head()

,day,roughness_m,head_loss_m,pressure_drop_Pa
0,0,0.000002,0.178394,1744.790537
1,1,0.000002,0.178392,1744.776016
2,2,0.000002,0.178418,1745.024928
3,3,0.000002,0.178424,1745.088268
4,4,0.000002,0.178394,1744.799668


In [3]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['day'], y=df['roughness_m']*1e6, mode='markers',
                          marker=dict(size=3, opacity=0.5), name='Synthetic roughness'))
fig.update_layout(title='Synthetic Roughness Degradation Over Time',
                   xaxis_title='Day', yaxis_title='Roughness (micrometers)',
                   template='plotly_white')
fig.show()

### Fit a Random Forest regressor and evaluate on held-out data

In [4]:
from src.machine_learning.degradation_model import fit_degradation_model, predict_roughness, predict_maintenance_threshold_day

result = fit_degradation_model(df['day'], df['roughness_m'])
print(f'Train R2: {result.train_r2:.4f}')
print(f'Test R2:  {result.test_r2:.4f}')
print(f'Test MAE: {result.test_mae:.3e} m')

Train R2: 0.9918
Test R2:  0.9335
Test MAE: 7.330e-08 m


In [5]:
days_smooth = np.arange(0, 730, 5)
predicted = predict_roughness(result, days_smooth)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['day'], y=df['roughness_m']*1e6, mode='markers',
                          marker=dict(size=3, opacity=0.3), name='Synthetic data'))
fig.add_trace(go.Scatter(x=days_smooth, y=predicted*1e6, mode='lines',
                          line=dict(color='red', width=3), name='Model prediction'))
fig.update_layout(title='Random Forest Fit to Synthetic Degradation Data',
                   xaxis_title='Day', yaxis_title='Roughness (micrometers)',
                   template='plotly_white')
fig.show()

### When should maintenance be scheduled?

Find the day the model predicts roughness will reach 1.5x its initial value — a simple illustrative maintenance trigger.

In [6]:
initial_roughness = df['roughness_m'].iloc[0]
threshold_day = predict_maintenance_threshold_day(
    result, roughness_threshold_m=initial_roughness * 1.5, max_day=730,
)

print(f'Initial roughness: {initial_roughness*1e6:.3f} micrometers')
print(f'Predicted day reaching 1.5x: day {threshold_day} (~{threshold_day/365:.2f} years)' 
      if threshold_day else 'Threshold not reached within search horizon')

Initial roughness: 1.514 micrometers
Predicted day reaching 1.5x: day 180 (~0.49 years)


**Limitation**: Random Forests don't extrapolate — predictions far beyond the training data's day range will plateau rather than continue the trend. This threshold search is only reliable within the training range (here, 0-730 days).

## Part 2: Anomaly Detection (Leak / Cavitation Onset)

### Generate synthetic sensor data with injected anomalies

In [7]:
from src.machine_learning.synthetic_data import generate_synthetic_sensor_data_with_anomalies

sensor_df = generate_synthetic_sensor_data_with_anomalies(
    base_pressure_Pa=300_000, base_flow_m3s=0.0005,
    n_samples=400, n_anomalies=20, anomaly_magnitude_fraction=0.3,
)
sensor_df.head()

,sample,pressure_Pa,flow_m3s,is_true_anomaly
0,0,301828.302479,0.000498,False
1,1,293760.095363,0.000502,False
2,2,304502.707175,0.000508,False
3,3,305643.388298,0.000496,False
4,4,288293.788868,0.000505,False


In [8]:
fig = go.Figure()
normal = sensor_df[~sensor_df['is_true_anomaly']]
anomalies = sensor_df[sensor_df['is_true_anomaly']]
fig.add_trace(go.Scatter(x=normal['sample'], y=normal['pressure_Pa'], mode='markers',
                          marker=dict(size=4, color='steelblue'), name='Normal'))
fig.add_trace(go.Scatter(x=anomalies['sample'], y=anomalies['pressure_Pa'], mode='markers',
                          marker=dict(size=8, color='red', symbol='x'), name='True anomaly'))
fig.update_layout(title='Synthetic Pressure Readings (ground truth shown)',
                   xaxis_title='Sample', yaxis_title='Pressure (Pa)', template='plotly_white')
fig.show()

### Approach A: SPC control chart (transparent, no training needed)

In [9]:
from src.machine_learning.anomaly_detection import (
    compute_spc_control_limits, flag_spc_anomalies, evaluate_detector_against_ground_truth,
)

# Establish baseline from the first 100 samples' normal (non-anomalous) readings
baseline_mask = ~sensor_df['is_true_anomaly'].iloc[:100]
baseline = sensor_df['pressure_Pa'].iloc[:100][baseline_mask]
limits = compute_spc_control_limits(baseline)
print(f'Center: {limits.center_line:.1f} Pa, UCL: {limits.upper_control_limit:.1f}, '
      f'LCL: {limits.lower_control_limit:.1f}')

spc_flags = flag_spc_anomalies(sensor_df['pressure_Pa'], limits)
spc_metrics = evaluate_detector_against_ground_truth(spc_flags, sensor_df['is_true_anomaly'])
print('SPC chart performance on this synthetic benchmark:', spc_metrics)

Center: 299711.0 Pa, UCL: 312315.0, LCL: 287107.1
SPC chart performance on this synthetic benchmark: {'precision': 0.6451612903225806, 'recall': 1.0, 'f1': 0.7843137254901961, 'n_true_positives': 20, 'n_false_positives': 11, 'n_false_negatives': 0}


### Approach B: Isolation Forest (ML, multivariate)

In [10]:
from src.machine_learning.anomaly_detection import fit_isolation_forest_detector, detect_anomalies

features = sensor_df[['pressure_Pa', 'flow_m3s']].values
model = fit_isolation_forest_detector(features, contamination=0.05)
iso_result = detect_anomalies(model, features)

iso_metrics = evaluate_detector_against_ground_truth(iso_result.anomaly_flags, sensor_df['is_true_anomaly'])
print('Isolation Forest performance on this synthetic benchmark:', iso_metrics)

Isolation Forest performance on this synthetic benchmark: {'precision': 0.85, 'recall': 0.85, 'f1': 0.85, 'n_true_positives': 17, 'n_false_positives': 3, 'n_false_negatives': 3}


## Part 3: ISO 14224 Reliability Taxonomy — CMMS Export

`iso14224.annotate_anomaly_flags` wraps the Isolation Forest output above in ISO 14224:2016 labels, producing the kind of structured record a Computerised Maintenance Management System (CMMS) would store — this is what "predictive maintenance" needs to look like to actually reach a Reliability Engineer's workflow, not just a data scientist's notebook.

This section's scenario is leak / cavitation onset (per the Part 2 header above), so we override `annotate_anomaly_flags`'s `failure_mechanism_candidate` default (`MECH_FOULING`, tuned for Part 1's roughness-growth scenario) with `MECH_CAVITATION` instead. The function's docstring is explicit that this is a *candidate* assignment you set based on your actual equipment and fluid chemistry — mechanism confirmation always requires physical inspection, ISO 14224 label or not.

**Still synthetic**: the anomalies labelled here are the injected, fabricated ones from Part 2. Real value comes from wiring this same function up to flags from an actual sensor pipeline. The Work Order ID, Asset ID, and timestamp columns below are mocked purely to show the CMMS-facing shape of the output — nothing here is a real work order.

In [11]:
from src.machine_learning.iso14224 import annotate_anomaly_flags, MECH_CAVITATION

records = annotate_anomaly_flags(
    iso_result.anomaly_flags,
    iso_result.anomaly_scores,
    failure_mechanism_candidate=MECH_CAVITATION,
)

n_flagged = sum(r.is_anomaly for r in records)
print(f'{n_flagged} of {len(records)} observations annotated as anomalous '
      f'(matches the {int(iso_result.anomaly_flags.sum())} flagged by Isolation Forest above).')

20 of 400 observations annotated as anomalous (matches the 20 flagged by Isolation Forest above).


In [12]:
import pandas as pd

# Human-readable text for the codes used in this demo — transcribed from
# the constants' doc-comments in iso14224.py purely for display. The
# module itself works with the short codes alone, which is what a real
# CMMS integration would actually exchange.
CODE_TEXT = {
    'AIR': 'Abnormal instrument reading',
    'CAV': 'Cavitation (vapour-bubble collapse damaging the pipe bore)',
    'AUT': 'Automatic protective / alarm system',
}


def describe(code):
    return f'{code} — {CODE_TEXT.get(code, code)}' if code else None


# A real CMMS shows confirmed events, not 400 mostly-empty rows — filter
# to flagged observations and rank by severity (most negative score =
# most anomalous, first), the way a reliability engineer would triage.
flagged = sorted((r for r in records if r.is_anomaly), key=lambda r: r.anomaly_score)

n = len(flagged)
tercile = max(n // 3, 1)


def priority(rank):
    if rank < tercile:
        return 'High'
    if rank < 2 * tercile:
        return 'Medium'
    return 'Low'


cmms_table = pd.DataFrame([
    {
        'work_order_id': f'WO-2026-{rank + 1:04d}',
        'detection_timestamp': pd.Timestamp('2026-01-01') + pd.Timedelta(hours=r.observation_index),
        'asset_id': 'PIPE-001',
        'failure_mode': describe(r.failure_mode),
        'failure_mechanism_candidate': describe(r.failure_mechanism_candidate),
        'detection_method': describe(r.detection_method),
        'anomaly_score': round(r.anomaly_score, 4),
        'priority': priority(rank),
    }
    for rank, r in enumerate(flagged)
])
cmms_table

,work_order_id,detection_timestamp,asset_id,failure_mode,failure_mechanism_candidate,detection_method,anomaly_score,priority
0,WO-2026-0001,2026-01-13 10:00:00,PIPE-001,AIR — Abnormal instrument reading,CAV — Cavitation (vapour-bubble collapse damag...,AUT — Automatic protective / alarm system,-0.7403,High
1,WO-2026-0002,2026-01-10 17:00:00,PIPE-001,AIR — Abnormal instrument reading,CAV — Cavitation (vapour-bubble collapse damag...,AUT — Automatic protective / alarm system,-0.7188,High
2,WO-2026-0003,2026-01-07 08:00:00,PIPE-001,AIR — Abnormal instrument reading,CAV — Cavitation (vapour-bubble collapse damag...,AUT — Automatic protective / alarm system,-0.7083,High
3,WO-2026-0004,2026-01-17 10:00:00,PIPE-001,AIR — Abnormal instrument reading,CAV — Cavitation (vapour-bubble collapse damag...,AUT — Automatic protective / alarm system,-0.6958,High
4,WO-2026-0005,2026-01-05 09:00:00,PIPE-001,AIR — Abnormal instrument reading,CAV — Cavitation (vapour-bubble collapse damag...,AUT — Automatic protective / alarm system,-0.6938,High
5,WO-2026-0006,2026-01-09 10:00:00,PIPE-001,AIR — Abnormal instrument reading,CAV — Cavitation (vapour-bubble collapse damag...,AUT — Automatic protective / alarm system,-0.6715,High
6,WO-2026-0007,2026-01-12 19:00:00,PIPE-001,AIR — Abnormal instrument reading,CAV — Cavitation (vapour-bubble collapse damag...,AUT — Automatic protective / alarm system,-0.6652,Medium
7,WO-2026-0008,2026-01-09 13:00:00,PIPE-001,AIR — Abnormal instrument reading,CAV — Cavitation (vapour-bubble collapse damag...,AUT — Automatic protective / alarm system,-0.6637,Medium
8,WO-2026-0009,2026-01-05 08:00:00,PIPE-001,AIR — Abnormal instrument reading,CAV — Cavitation (vapour-bubble collapse damag...,AUT — Automatic protective / alarm system,-0.6583,Medium
9,WO-2026-0010,2026-01-17 09:00:00,PIPE-001,AIR — Abnormal instrument reading,CAV — Cavitation (vapour-bubble collapse damag...,AUT — Automatic protective / alarm system,-0.6529,Medium


A table like this — or a version of it posted to your CMMS's work-order API — is the actual deliverable of a predictive-maintenance pipeline in an operations context; a precision/recall number alone doesn't reach a technician's worklist. The `priority` banding above is a plain tercile split on anomaly score for illustration only — a real deployment would set thresholds from your organization's risk matrix (e.g. ISO 14224 §7's consequence categories), not an arbitrary 1/3 split.

## Takeaways

- **SPC control charts** need no training, are fully transparent (anyone can audit the UCL/LCL math), and integrate naturally with this project's existing Lean/Poka-Yoke approach — prefer these when interpretability matters or historical data is limited.
- **Isolation Forest** can incorporate multiple sensor signals at once (here, pressure *and* flow jointly) and may catch subtler multivariate patterns SPC would miss, at the cost of being a black box.
- Both numbers above are evaluated against *known* injected anomalies — a luxury real deployments don't have. Precision/recall here describe performance on this synthetic benchmark only, not a guarantee of real-world performance.
- **ISO 14224 annotation** turns a bare boolean flag array into a structured, CMMS-shaped record (failure mode, candidate mechanism, detection method, standard reference) — the difference between "the model said anomaly" and something a reliability engineer can act on, audit, and file against a standard.